In [ ]:
# Import required libraries
import os
import csv
import duckdb
import requests
import re
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

# Establish connection to DuckDB database
db_path = "E:/526-Data-Warehousing/Duckdb/imdb_cleaned.duckdb"
con = duckdb.connect(db_path)

print("Connected to DuckDB successfully.")

Connected to DuckDB!


In [ ]:
# Define the columns to retain for each dataset
columns_to_keep = {
    "title.akas.csv": ["titleId", "title", "region", "language", "types", "attributes", "isOriginalTitle"],
    "title.episode.csv": ["tconst", "parentTconst", "seasonNumber", "episodeNumber"],
    "name.basics.csv": ["nconst", "primaryName", "birthYear", "deathYear", "primaryProfession", "knownForTitles"],
    "title.basics.csv": ["tconst", "titleType", "primaryTitle", "originalTitle", "isAdult", "startYear", "endYear", "runtimeMinutes", "genres"],
    "title.ratings.csv": ["tconst", "averageRating", "numVotes"],
    "title.principals.csv": ["tconst", "ordering", "nconst", "category", "job", "characters"],
    "title.crew.csv": ["tconst", "directors", "writers"]
}

# Specify input and output directories
input_folder = r"E:/526-Data-Warehousing/Duckdb/imdb_datasets"
output_folder = r"E:/526-Data-Warehousing/Duckdb/cleaned"
os.makedirs(output_folder, exist_ok=True)

# Define cleaning function to standardize missing values
def clean_value(x):
    if x is None:
        return '\\N'
    x = x.strip()
    return x if x else '\\N'

# Process and clean each file
for file_name, keep_cols in columns_to_keep.items():
    input_path = os.path.join(input_folder, file_name)
    output_path = os.path.join(output_folder, file_name.replace(".csv", ".cleaned.csv"))

    print(f"Processing: {file_name}")
    try:
        with open(input_path, 'r', encoding='utf-8') as infile, open(output_path, 'w', newline='', encoding='utf-8') as outfile:
            reader = csv.DictReader(infile)
            writer = csv.DictWriter(outfile, fieldnames=keep_cols)
            writer.writeheader()

            for row in reader:
                cleaned_row = {col: clean_value(row.get(col, '')) for col in keep_cols}
                writer.writerow(cleaned_row)

        print(f"Cleaned and saved: {output_path}")
    except Exception as e:
        print(f"Error cleaning {file_name}: {e}")

🔄 Cleaning: title.akas.csv
✅ Cleaned and saved: E:/526-Data-Warehousing/Duckdb/cleaned\title.akas.cleaned.csv

🔄 Cleaning: title.episode.csv
✅ Cleaned and saved: E:/526-Data-Warehousing/Duckdb/cleaned\title.episode.cleaned.csv

🔄 Cleaning: name.basics.csv
✅ Cleaned and saved: E:/526-Data-Warehousing/Duckdb/cleaned\name.basics.cleaned.csv

🔄 Cleaning: title.basics.csv
✅ Cleaned and saved: E:/526-Data-Warehousing/Duckdb/cleaned\title.basics.cleaned.csv

🔄 Cleaning: title.ratings.csv
✅ Cleaned and saved: E:/526-Data-Warehousing/Duckdb/cleaned\title.ratings.cleaned.csv

🔄 Cleaning: title.principals.csv
✅ Cleaned and saved: E:/526-Data-Warehousing/Duckdb/cleaned\title.principals.cleaned.csv

🔄 Cleaning: title.crew.csv
✅ Cleaned and saved: E:/526-Data-Warehousing/Duckdb/cleaned\title.crew.cleaned.csv



In [ ]:
# Define mapping of table names to cleaned CSV files
tables = {
    "title_basics": "title.basics.cleaned.csv",
    "name_basics": "name.basics.cleaned.csv",
    "title_ratings": "title.ratings.cleaned.csv",
    "title_akas": "title.akas.cleaned.csv",
    "title_crew": "title.crew.cleaned.csv",
    "title_episode": "title.episode.cleaned.csv",
    "title_principals": "title.principals.cleaned.csv"
}

# Load each cleaned CSV file into DuckDB as a table
for table, file in tables.items():
    con.execute(f"""
        CREATE OR REPLACE TABLE {table} AS
        SELECT * FROM read_csv_auto('{os.path.join(output_folder, file)}', HEADER=TRUE, nullstr='\\N');
    """)

print("Tables loaded into DuckDB successfully.")

✅ Tables loaded into DuckDB!


In [ ]:
# Create a view combining movies and directors
con.execute("""
CREATE OR REPLACE VIEW movie_director_view AS
SELECT 
    t.tconst,
    t.primaryTitle,
    t.startYear,
    p.primaryName AS director_name
FROM title_basics t
JOIN title_crew c ON t.tconst = c.tconst
JOIN name_basics p ON c.directors LIKE '%' || p.nconst || '%';
""")

# Create a view combining movies and ratings
con.execute("""
CREATE OR REPLACE VIEW movie_rating_view AS
SELECT 
    t.tconst,
    t.primaryTitle,
    t.startYear,
    r.averageRating,
    r.numVotes
FROM title_basics t
JOIN title_ratings r ON t.tconst = r.tconst;
""")

# Create a view linking actors to their known movie titles
con.execute("""
CREATE OR REPLACE VIEW actor_known_titles_view AS
SELECT 
    n.nconst,
    n.primaryName,
    t.primaryTitle
FROM name_basics n
JOIN title_basics t ON n.knownForTitles LIKE '%' || t.tconst || '%';
""")

print("Views created successfully.")

✅ Views created successfully!


In [16]:
# Function to extract clean SQL code from DeepSeek output
def extract_sql_code(text):
    matches = re.findall(r"```sql(.*?)```", text, re.DOTALL | re.IGNORECASE)
    return matches[0].strip() if matches else text.strip()

# Function to send user question to DeepSeek and clean the SQL output
def ask_deepseek(user_question):
    schema_prompt = """
You are a SQL assistant working with a DuckDB database containing:

- title_basics
- name_basics
- title_ratings
- title_crew
- title_principals
- title_episode
- title_akas
- movie_director_view
- movie_rating_view
- actor_known_titles_view

Notes:
- Match titles case-insensitively using LOWER().
- Always handle NULL values using COALESCE.
- Match directors, writers, knownForTitles fields using LIKE because they contain comma-separated IDs.
- Genres are stored as plain comma-separated text, not arrays. Use LIKE for genre matching. Do NOT use UNNEST or str_split.
- Use the table names exactly as they are defined in the database.
- Use the column names exactly as they are defined in the database.
- Write SQL queries using the SQL dialect and syntax supported by DuckDB.
"""

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "deepseek/deepseek-chat-v3-0324:free",
        "temperature": 0.0,
        "messages": [
            {"role": "user", "content": schema_prompt},
            {"role": "user", "content": user_question}
        ]
    }

    response = requests.post("https://openrouter.ai/api/v1/chat/completions", headers=headers, json=data)

    if response.status_code != 200:
        print("API Error:", response.text)
        return None

    sql_raw = response.json()["choices"][0]["message"]["content"]
    sql_clean = extract_sql_code(sql_raw)

    # Auto-correct common mistakes in generated SQL
    replacements = {
    "isOriginalTitle = 1": "COALESCE(isOriginalTitle, '') = '1'",
    "title_principals.job": "title_principals.category",
    "T1.titleId": "T1.tconst",
    "t2.title": "t2.primaryTitle",
    "md.directors": "md.director_name",
    "d.directors": "d.director_name"
    }

    for wrong, correct in replacements.items():
        if wrong in sql_clean:
            print(f"Auto-corrected: '{wrong}' to '{correct}'")
        sql_clean = sql_clean.replace(wrong, correct)

    return sql_clean

In [18]:
# Accept user question input
user_question = input("Enter your IMDb-related question: ")

# Generate SQL using DeepSeek
sql_query = ask_deepseek(user_question)

# Execute the generated SQL query
if sql_query:
    print("\nGenerated SQL:\n", sql_query)

    try:
        print("\nExecuting SQL :-")
        result = con.execute(sql_query)
        rows = result.fetchall()

        if not rows:
            print("No results found.")
        else:
            columns = [desc[0] for desc in result.description]
            print("Columns:", columns)
            for row in rows[:10]:
                print(row)

    except Exception as e:
        print("SQL Execution Error:", e)
else:
    print("No SQL generated.")


Generated SQL:
 SELECT 
    n.nconst AS id,
    n.primaryName AS name,
    n.birthYear,
    n.deathYear,
    n.primaryProfession,
    n.knownForTitles
FROM 
    name_basics n
WHERE 
    LOWER(n.primaryName) = 'robert downey jr.';

Executing SQL :-
No results found.
